# 16. Recuperación híbrida: BM25 + cada codificador

Ejecutar en `07_RAG/Pruebas_RAG/`. Lee los dos JSON existentes y genera **solo** `hibrida_rrf_resultados_v1.json`. No modifica entradas ni calcula embeddings.

Se fija `c = 60`, peso igual y las listas completas disponibles: 519 posiciones densas y únicamente los documentos con BM25 > 0. Un artículo sin coincidencia léxica recibe aporte cero de BM25. Se conservan todos los identificadores y las formulaciones ES/EN por separado.

RRF suma `1 / (c + posición)` por lista; **c no es el corte k de Recall@k**. Esta configuración es una referencia fija, no un óptimo demostrado. No se usan artículos orientadores ni juicios de pertinencia.

Referencia: Cormack, Clarke y Büttcher (2009), *Reciprocal rank fusion outperforms Condorcet and individual rank learning methods*. DOI: `10.1145/1571941.1572114`.


### 1. Entradas

In [1]:
import hashlib, json, math, platform, sys
from pathlib import Path
from time import perf_counter
from statistics import median
from datetime import datetime, timezone
import pandas as pd

C_RRF = 60
REPETICIONES = 10
MODELOS = {"bge_m3", "qwen3_0_6b", "qwen3_4b", "harrier_0_6b", "nemotron3_1b",
           "granite_311m_r2", "pplx_0_6b", "arctic_l_v2", "embeddinggemma2_text",
           "qwen3_vl_2b_text", "pplx_4b_bf16"}

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "07_RAG/Pruebas_RAG/densa_resultados_v1.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta dentro de Tesis_Multimodelo.")
CARPETA = REPO / "07_RAG/Pruebas_RAG"
ENTRADAS = {"bm25": CARPETA / "bm25_resultados_v1.json",
            "densa": CARPETA / "densa_resultados_v1.json"}
SALIDA = CARPETA / "hibrida_rrf_resultados_v1.json"

def sha_json(x):
    texto = json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

def sha_archivo(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def verificar(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)

huellas = {k: sha_archivo(p) for k, p in ENTRADAS.items()}
bm25 = json.loads(ENTRADAS["bm25"].read_text(encoding="utf-8"))
densa = json.loads(ENTRADAS["densa"].read_text(encoding="utf-8"))
print("Entradas cargadas. No se cargarán codificadores ni índices FAISS.")


Entradas cargadas. No se cargarán codificadores ni índices FAISS.


### 2. Verificación de resultados

In [2]:
verificar(bm25["version_esquema"] == "BM25_RESULTADOS_V1", "Esquema BM25 distinto.")
verificar(densa["version_esquema"] == "RECUPERACION_DENSA_V1", "Esquema denso distinto.")
verificar(set(densa["sistemas"]) == MODELOS, "Deben estar los 11 modelos completos.")
verificar(bm25["configuracion"]["campo_documento"] == "abstract", "BM25 no usa abstract.")
fuentes = densa["fuentes"]
for campo in ("catalogo", "consultas"):
    verificar(bm25["fuentes"][campo]["sha256"] == fuentes[f"sha256_{campo}"],
              f"Versiones diferentes de {campo}.")
verificar(sha_json(bm25["resultados"]) == bm25["sha256_resultados"], "BM25 fue alterado.")
consultas = densa["consultas"]
qids = [q["query_id"] for q in consultas]
verificar(len(qids) == len(set(qids)) == 24, "Se esperan 24 consultas distintas.")
verificar([r["query_id"] for r in bm25["resultados"]] == qids, "Consultas BM25 diferentes.")
for q, r in zip(consultas, bm25["resultados"]):
    verificar(all(r[k] == q[k] for k in q), "Texto, idioma o división de consulta distintos.")
ids = {d["article_id"] for d in bm25["resultados"][0]["documentos"]}
verificar(len(ids) == 519, "Se esperan 519 artículos.")
verificar(len({int(i) for i in ids}) == 519, "Identificadores numéricos ambiguos.")

def validar_lista(docs, campo, lexico=False):
    verificar(len(docs) == 519 and {d["article_id"] for d in docs} == ids,
              "Faltan artículos o hay identificadores duplicados.")
    verificar(all(math.isfinite(d[campo]) for d in docs), "Puntuaciones no finitas.")
    verificar(docs == sorted(docs, key=lambda d: (-d[campo], int(d["article_id"]))),
              "Orden o desempate incorrectos.")
    for posicion, d in enumerate(docs, 1):
        verificar(d[campo] >= 0 if lexico else abs(d[campo]) <= 1.00001,
                  "Puntuación fuera de rango.")
        esperado = None if lexico and d[campo] == 0 else posicion
        verificar(d["rank"] == esperado, "Posición incompatible con la puntuación.")

for r in bm25["resultados"]:
    validar_lista(r["documentos"], "score_bm25", True)
for clave, s in densa["sistemas"].items():
    verificar(sha_json(s["resultados"]) == s["sha256_resultados"], f"Resultados alterados: {clave}")
    verificar([r["query_id"] for r in s["resultados"]] == qids, f"Consultas incompletas: {clave}")
    verificar(s["condicion"]["campo_documento"] == "abstract", f"Campo distinto: {clave}")
    verificar(s["referencia_bm25"]["sha256_resultados"] == bm25["sha256_resultados"],
              f"Referencia BM25 diferente: {clave}")
    for r in s["resultados"]:
        validar_lista(r["documentos"], "score_coseno")
        v = r["vector_consulta"]
        verificar(len(v) == s["condicion"]["dimension"] and all(math.isfinite(x) for x in v),
                  f"Vector inválido: {clave}")
        verificar(abs(math.sqrt(sum(x*x for x in v)) - 1) < 1e-5, f"Vector sin L2: {clave}")

configuracion = {"metodo": "RRF", "c": C_RRF, "pesos": {"bm25": 1, "densa": 1},
    "profundidad_densa": 519, "profundidad_bm25": "todos con score_bm25 > 0",
    "aporte_bm25_sin_coincidencia": 0, "umbral_coseno": None,
    "empates": "score_rrf descendente; article_id entero ascendente",
    "unidad_documental": "article_id; sin fusionar resúmenes idénticos",
    "qrels_utilizados": False, "metricas_calidad_calculadas": False}
firma = sha_json({"configuracion": configuracion, "fuentes": fuentes,
    "bm25": bm25["sha256_resultados"],
    "densa": {k: {"resultados": s["sha256_resultados"], "condicion": s["firma_condicion"]}
              for k, s in densa["sistemas"].items()}})
print("Validación: 11 modelos, 24 consultas por modelo y 519 artículos por consulta.")


Validación: 11 modelos, 24 consultas por modelo y 519 artículos por consulta.


### 3. Fusión de las 11 parejas

In [3]:
def fusionar(docs_bm25, docs_densos):
    posiciones_bm25 = {d["article_id"]: d["rank"] for d in docs_bm25 if d["rank"] is not None}
    fusion = []
    for d in docs_densos:
        rb = posiciones_bm25.get(d["article_id"])
        rd = d["rank"]
        score = 1 / (C_RRF + rd) + (1 / (C_RRF + rb) if rb is not None else 0.0)
        fusion.append({"article_id": d["article_id"], "score_rrf": score,
                       "rank_bm25": rb, "rank_denso": rd})
    fusion.sort(key=lambda d: (-d["score_rrf"], int(d["article_id"])))
    for rank, d in enumerate(fusion, 1):
        d["rank"] = rank
    return fusion

reutilizado = SALIDA.exists()
if reutilizado:
    salida = json.loads(SALIDA.read_text(encoding="utf-8"))
    verificar(salida["firma_experimento"] == firma, "La salida corresponde a otra condición; no se sobrescribe.")
    verificar(salida["configuracion"] == configuracion and salida["fuentes"] == fuentes
              and salida["consultas"] == consultas, "Metadatos de salida diferentes.")
    verificar(set(salida["sistemas"]) == {f"rrf_bm25_{k}" for k in MODELOS}, "Salida incompleta.")
    print("Resultado existente: se comprobará sin repetir las mediciones.")
else:
    salida = {"version_esquema": "RECUPERACION_HIBRIDA_RRF_V1", "firma_experimento": firma,
        "configuracion": configuracion, "fuentes": fuentes,
        "archivos_entrada": {k: {"archivo": p.name, "sha256": huellas[k]} for k, p in ENTRADAS.items()},
        "consultas": consultas, "sistemas": {},
        "advertencias": [
            "No hay juicios humanos de pertinencia; estos resultados no seleccionan un ganador.",
            "Qwen3-0.6B: revisión de los artículos no registrada; snapshot de consultas inferido de caché.",
            "Qwen3-0.6B: prefijo_consulta quedó vacío en el JSON, aunque la libreta 15 usa prompt_name='query'.",
            "En Qwen3-0.6B, tokens_consulta se contó sin el prefijo. No utilizar ese conteo como longitud total.",
            "RRF conserva los rankings recibidos; no corrige ni verifica de nuevo la inferencia de los codificadores."],
        "ejecucion": {"fecha_utc": datetime.now(timezone.utc).isoformat(),
            "python": sys.version.split()[0], "sistema": platform.platform(),
            "calentamientos_por_consulta": 1, "repeticiones_por_consulta": REPETICIONES,
            "alcance_latencia": "Solo fusión y ordenamiento con listas en RAM; no latencia completa de recuperación",
            "gpu_utilizada": False, "ram_pico_bytes": None}}
    for clave, s in densa["sistemas"].items():
        resultados, latencias = [], []
        for b, r in zip(bm25["resultados"], s["resultados"]):
            fusionar(b["documentos"], r["documentos"])
            tiempos = []
            for repeticion in range(REPETICIONES):
                t0 = perf_counter()
                docs = fusionar(b["documentos"], r["documentos"])
                tiempos.append(1000 * (perf_counter() - t0))
                if repeticion == 0:
                    primero = docs
                else:
                    verificar(docs == primero, "La fusión cambió entre repeticiones.")
            resultados.append({"query_id": r["query_id"], "documentos": primero})
            latencias.append({"query_id": r["query_id"], "muestras_ms": tiempos, "mediana_ms": median(tiempos)})
        salida["sistemas"][f"rrf_bm25_{clave}"] = {
            "codificador": clave, "revision_procedencia": s["condicion"].get("revision_procedencia", "fijada_en_npz"),
            "sha256_resultados_bm25": bm25["sha256_resultados"],
            "sha256_resultados_densos": s["sha256_resultados"], "firma_condicion_densa": s["firma_condicion"],
            "resultados": resultados, "sha256_resultados": sha_json(resultados), "latencias_fusion": latencias}
        print(f"{clave}: 24 fusiones completas")


bge_m3: 24 fusiones completas
granite_311m_r2: 24 fusiones completas
harrier_0_6b: 24 fusiones completas
nemotron3_1b: 24 fusiones completas
arctic_l_v2: 24 fusiones completas
embeddinggemma2_text: 24 fusiones completas
pplx_0_6b: 24 fusiones completas
qwen3_vl_2b_text: 24 fusiones completas
qwen3_4b: 24 fusiones completas
pplx_4b_bf16: 24 fusiones completas
qwen3_0_6b: 24 fusiones completas


### 4. Guardado y comprobación

In [4]:
for clave, s in salida["sistemas"].items():
    verificar(sha_json(s["resultados"]) == s["sha256_resultados"], f"Salida alterada: {clave}")
    verificar([r["query_id"] for r in s["resultados"]] == qids, f"Salida incompleta: {clave}")
    fuente_densa = densa["sistemas"][s["codificador"]]["resultados"]
    for b, d, r in zip(bm25["resultados"], fuente_densa, s["resultados"]):
        verificar(r["documentos"] == fusionar(b["documentos"], d["documentos"]), "Fusión no reproducible.")
for clave, p in ENTRADAS.items():
    verificar(sha_archivo(p) == huellas[clave], "Una entrada cambió durante la ejecución.")

if not reutilizado:
    libreta = CARPETA / "16_recuperacion_hibrida_rrf.ipynb"
    if libreta.is_file():
        celdas = json.loads(libreta.read_text(encoding="utf-8"))["cells"]
        salida["ejecucion"]["sha256_codigo"] = sha_json(["".join(c["source"]) for c in celdas if c["cell_type"] == "code"])
    else:
        raise FileNotFoundError("Guarda la libreta con el nombre 16_recuperacion_hibrida_rrf.ipynb.")
    temporal = SALIDA.with_suffix(".json.tmp")
    if SALIDA.exists() or temporal.exists():
        raise FileExistsError("Ya existe una salida o escritura temporal; no se reemplaza.")
    with temporal.open("x", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
        f.write("\n")
    if SALIDA.exists():
        raise FileExistsError("Apareció otra salida durante la escritura; se conserva el temporal.")
    temporal.rename(SALIDA)

resumen = [{"sistema": k, "consultas": len(s["resultados"]),
            "puntuaciones": sum(len(r["documentos"]) for r in s["resultados"]),
            "mediana_fusion_ms": round(median(t["mediana_ms"] for t in s["latencias_fusion"]), 4)}
           for k, s in salida["sistemas"].items()]
display(pd.DataFrame(resumen))
print("11 híbridos | 264 listas | 137 016 puntuaciones")
print("Qwen3-0.6B conserva las advertencias de trazabilidad en el JSON.")
print("Archivo:", SALIDA)
print("Sin métricas de pertinencia. Siguiente actividad: preparar los documentos para anotación.")


,sistema,consultas,puntuaciones,mediana_fusion_ms
0,rrf_bm25_bge_m3,24,12456,0.4912
1,rrf_bm25_granite_311m_r2,24,12456,0.4838
2,rrf_bm25_harrier_0_6b,24,12456,0.4932
3,rrf_bm25_nemotron3_1b,24,12456,0.4929
4,rrf_bm25_arctic_l_v2,24,12456,0.4906
5,rrf_bm25_embeddinggemma2_text,24,12456,0.4922
6,rrf_bm25_pplx_0_6b,24,12456,0.4992
7,rrf_bm25_qwen3_vl_2b_text,24,12456,0.4962
8,rrf_bm25_qwen3_4b,24,12456,0.4955
9,rrf_bm25_pplx_4b_bf16,24,12456,0.4988


11 híbridos | 264 listas | 137 016 puntuaciones
Qwen3-0.6B conserva las advertencias de trazabilidad en el JSON.
Archivo: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\hibrida_rrf_resultados_v1.json
Sin métricas de pertinencia. Siguiente actividad: preparar los documentos para anotación.
